# Amenity Data Collection
This notebook collects and geocodes all amenity reference data required for geospatial feature engineering. Each section produces a processed CSV that will be used in the next notebook to calculate distances from HDB blocks to nearby amenities.

## Setup
Import required libraries, load the OneMap API token from the environment, and load the MOE school directory.

Schools are geocoded by postal code rather than address since postal codes are unambiguous and return more reliable results from the OneMap search API.

In [50]:
import pandas as pd
import numpy as np
import requests
import time
import os
from dotenv import load_dotenv

load_dotenv('../.env')
token = os.getenv('ONEMAP_TOKEN')

print("Token loaded:", token is not None)

Token loaded: True


In [51]:
schools = pd.read_csv('../data/raw/schools.csv')
print(schools.shape)
print("\nmainlevel_code values:")
print(schools['mainlevel_code'].value_counts())

(337, 31)

mainlevel_code values:
mainlevel_code
PRIMARY                         179
SECONDARY (S1-S5)               117
SECONDARY (S1-S4)                16
JUNIOR COLLEGE                   10
MIXED LEVEL (S1-JC2)             10
MIXED LEVEL (P1-S4)               3
CENTRALISED INSTITUTE             1
MIXED LEVEL (S1-S5, JC1-JC2)      1
Name: count, dtype: int64


## Geocoding Function (Postal Code)
The `geocode_postal` function queries the OneMap search API with a postal code, returning latitude and longitude coordinates. It uses the same authenticated token and retry logic as the address geocoding function in notebook 04.

In [4]:
def geocode_postal(postal_code, token, retries=3):
    url = "https://www.onemap.gov.sg/api/common/elastic/search"
    params = {
        "searchVal": str(postal_code),
        "returnGeom": "Y",
        "getAddrDetails": "Y",
        "pageNum": 1
    }
    headers = {"Authorization": token}
    for attempt in range(retries):
        try:
            response = requests.get(url, params=params, headers=headers, timeout=10)
            data = response.json()
            if data.get('found', 0) > 0:
                result = data['results'][0]
                return float(result['LATITUDE']), float(result['LONGITUDE'])
            else:
                time.sleep(1)
        except Exception as e:
            time.sleep(1)
    return None, None

row = schools.iloc[0]
lat, lon = geocode_postal(row['postal_code'], token)
print(f"School: {row['school_name']}")
print(f"Postal: {row['postal_code']}")
print(f"Coords: {lat}, {lon}")

School: ADMIRALTY PRIMARY SCHOOL
Postal: 738907
Coords: 1.4426347903311, 103.800040119743


## Geocode Schools
We geocode all 337 schools using their postal codes and save the results with their `mainlevel_code` for filtering in the distance calculation notebook.

`mainlevel_code` distinguishes PRIMARY, SECONDARY, JUNIOR COLLEGE and mixed-level schools, allowing us to compute separate features for `dist_nearest_school` and `dist_nearest_primary_school`.

In [5]:
results = []

for idx, row in schools.iterrows():
    lat, lon = geocode_postal(row['postal_code'], token)
    results.append({
        'school_name': row['school_name'],
        'postal_code': row['postal_code'],
        'mainlevel_code': row['mainlevel_code'],
        'latitude': lat,
        'longitude': lon
    })
    time.sleep(0.25)

schools_geocoded = pd.DataFrame(results)
print(f"Done. Geocoded {len(schools_geocoded):,} schools.")
print(f"Failed: {schools_geocoded['latitude'].isna().sum()}")

schools_geocoded.to_csv('../data/processed/schools_geocoded.csv', index=False)
print("Saved to data/processed/schools_geocoded.csv")

Done. Geocoded 337 schools.
Failed: 0
Saved to data/processed/schools_geocoded.csv


## Hawker Centres
Hawker centre data is downloaded from data.gov.sg as a GeoJSON file published by NEA. Coordinates are embedded directly in the GeoJSON geometry, so no geocoding is required.

We retain `est_completion_date` for each centre alongside its status. This is used in the distance calculation notebook to exclude centres that were not yet open at the time of a transaction.

Centres with `status == 'Under Construction'` are filtered out since they have no completion date and were not operational during the dataset period.

In [7]:
import json

with open('../data/raw/hawker_centres.geojson', 'r') as f:
    hawker_data = json.load(f)

hawkers = []
for feature in hawker_data['features']:
    props = feature['properties']
    coords = feature['geometry']['coordinates']
    hawkers.append({
        'name': props['NAME'],
        'longitude': coords[0],
        'latitude': coords[1],
        'status': props['STATUS'],
        'est_completion_date': props['EST_ORIGINAL_COMPLETION_DATE']
    })

hawkers_df = pd.DataFrame(hawkers)
print(f"Total hawker centres: {len(hawkers_df)}")
print(hawkers_df['status'].value_counts())
hawkers_df.head()

Total hawker centres: 129
status
Existing                  103
Existing (new)             16
Under Construction          6
Existing (replacement)      3
Interim Centre              1
Name: count, dtype: int64


,name,longitude,latitude,status,est_completion_date
0,Commonwealth Crescent Market,103.800367,1.306900,Existing,1965
1,Tiong Bahru Market,103.832182,1.284786,Existing,NaN
2,Golden Mile Food Centre,103.863878,1.303142,Existing,1975
3,Yew Tee Hawker Centre,103.745922,1.397185,Under Construction,2027
4,Dunman Food Centre,103.901825,1.309418,Existing,1974


In [8]:
# Filter out centres under construction
hawkers_existing = hawkers_df[hawkers_df['status'] != 'Under Construction'].copy()
print(f"Hawker centres after filtering: {len(hawkers_existing)}")

hawkers_existing.to_csv('../data/processed/hawker_centres.csv', index=False)
print("Saved to data/processed/hawker_centres.csv")

Hawker centres after filtering: 123
Saved to data/processed/hawker_centres.csv


## Geocoding Function (Name Query)
The `geocode_query` function accepts a free-text search string rather than a postal code. It is used for amenities that do not have postal codes readily available, such as MRT stations and malls.

In [52]:
def geocode_query(query, token, retries=3):
    url = "https://www.onemap.gov.sg/api/common/elastic/search"
    params = {
        "searchVal": query,
        "returnGeom": "Y",
        "getAddrDetails": "Y",
        "pageNum": 1
    }
    headers = {"Authorization": token}
    for attempt in range(retries):
        try:
            response = requests.get(url, params=params, headers=headers, timeout=10)
            data = response.json()
            if data.get('found', 0) > 0:
                result = data['results'][0]
                return float(result['LATITUDE']), float(result['LONGITUDE'])
            else:
                time.sleep(1)
        except Exception as e:
            time.sleep(1)
    return None, None

# Test on one station
lat, lon = geocode_query("Admiralty MRT Station Singapore", token)
print(f"Admiralty MRT: {lat}, {lon}")

Admiralty MRT: 1.44058856161847, 103.800990519771



## MRT Stations
MRT station data is sourced from a manually curated CSV based on the Wikipedia list of Singapore MRT stations. The list includes station name, station code, line, and opening date.

Opening dates are retained for the same reason as hawker centres — to exclude stations that did not yet exist at the time of a transaction. 

Stations are geocoded using a query of the form `"{station_name} MRT Station ({station_code})"`. Including the station code disambiguates stations with similar names and eliminates duplicate coordinate matches that occurred when querying by name alone.

In [53]:
mrt = pd.read_csv('../data/raw/mrt_stations.csv')
print(f"MRT stations loaded: {len(mrt)}")
mrt.head()

MRT stations loaded: 143


,station_name,station_code,line,opening_date
0,Admiralty,NS10,North-South Line,1996-02-10
1,Aljunied,EW9,East-West Line,1989-11-04
2,Ang Mo Kio,NS16,North-South Line,1987-11-07
3,Bartley,CC12,Circle Line,2009-05-28
4,Bayfront,CE1/DT16,Circle Line Extension/Downtown Line,2012-01-14


In [15]:
results = []

for idx, row in mrt.iterrows():
    # Include station code in query to disambiguate
    # Use first code only for multi-code stations
    primary_code = row['station_code'].split('/')[0]
    query = f"{row['station_name']} MRT Station ({primary_code})"
    lat, lon = geocode_query(query, token)
    results.append({
        'station_name': row['station_name'],
        'station_code': row['station_code'],
        'line': row['line'],
        'opening_date': row['opening_date'],
        'latitude': lat,
        'longitude': lon
    })
    time.sleep(0.25)

mrt_geocoded = pd.DataFrame(results)
print(f"Done. Geocoded {len(mrt_geocoded)} stations.")
print(f"Failed: {mrt_geocoded['latitude'].isna().sum()}")

# Check for duplicates
dupes = mrt_geocoded[mrt_geocoded.duplicated(subset=['latitude', 'longitude'], keep=False)]
print(f"Duplicate coordinates: {len(dupes)}")

Done. Geocoded 143 stations.
Failed: 0
Duplicate coordinates: 0


In [17]:
mrt_geocoded.to_csv('../data/processed/mrt_stations_geocoded.csv', index=False)
print("Saved to data/processed/mrt_stations_geocoded.csv")

Saved to data/processed/mrt_stations_geocoded.csv


## Malls
Mall data is sourced from the Wikipedia list of shopping malls in Singapore, organised by region. The list is comprehensive and covers all major retail destinations across the island.

Several malls failed to geocode under their Wikipedia names due to naming differences in the OneMap database. These are resolved using a name corrections dictionary that maps Wikipedia names to their OneMap equivalents. Raw mall names in the source CSV are preserved with corrections applied programmatically at geocoding time so the raw data remains traceable to its source.

In [18]:
malls = pd.read_csv('../data/raw/malls.csv')
print(f"Malls loaded: {len(malls)}")

results = []
for idx, row in malls.iterrows():
    query = f"{row['mall_name']} Singapore"
    lat, lon = geocode_query(query, token)
    results.append({
        'mall_name': row['mall_name'],
        'latitude': lat,
        'longitude': lon
    })
    time.sleep(0.25)

malls_geocoded = pd.DataFrame(results)
print(f"Done. Geocoded {len(malls_geocoded)} malls.")
print(f"Failed: {malls_geocoded['latitude'].isna().sum()}")
malls_geocoded.to_csv('../data/processed/malls_geocoded.csv', index=False)

Malls loaded: 116
Done. Geocoded 116 malls.
Failed: 10


In [19]:
failed_malls = malls_geocoded[malls_geocoded['latitude'].isna()]
print(failed_malls['mall_name'].tolist())

['Holland Village Shopping Mall', 'Shaw House and Centre', 'Tekka Centre', 'Novena Square Shopping Mall', 'i12 Katong', 'Heartland Mall', 'Lot 1', 'Yew Tee Point', '313@Somerset', 'Velocity@Novena']


In [30]:
name_corrections = {
    'Holland Village Shopping Mall': 'Holland Road Shopping Centre Singapore',
    'Shaw House and Centre': 'Shaw House Singapore',
    'Tekka Centre': 'Tekka Market Singapore',
    'Novena Square Shopping Mall': 'Novena Square Singapore',
    'i12 Katong': '112 East Coast Road Singapore',
    'Heartland Mall': 'Heartland Mall Kovan Singapore',
    'Lot 1': "Lot One, Shoppers' Mall",
    'Yew Tee Point': 'YewTee Point Singapore',
    '313@Somerset': '313 Somerset Singapore',
    'Velocity@Novena': 'Velocity Novena Singapore',
}

for idx, row in malls_geocoded[malls_geocoded['latitude'].isna()].iterrows():
    corrected_name = name_corrections.get(row['mall_name'], row['mall_name'])
    lat, lon = geocode_query(corrected_name, token)
    malls_geocoded.loc[idx, 'latitude'] = lat
    malls_geocoded.loc[idx, 'longitude'] = lon
    time.sleep(0.25)

print(f"Failed after correction: {malls_geocoded['latitude'].isna().sum()}")
print(malls_geocoded[malls_geocoded['latitude'].isna()]['mall_name'].tolist())

Failed after correction: 0
[]


In [31]:
malls_geocoded.to_csv('../data/processed/malls_geocoded.csv', index=False)
print("Saved to data/processed/malls_geocoded.csv")

Saved to data/processed/malls_geocoded.csv


## Expressways
Expressway geometries are sourced from the Singapore Land Authority's National Map Line dataset on data.gov.sg. This dataset contains road features as GeoJSON LineStrings, categorised by `FOLDERPATH`.

We extract all features where `FOLDERPATH == 'Layers/Expressway'`, which corresponds to the official SLA classification of expressway roads. This includes all 10 major expressways as well as underpasses managed under the same classification. Rather than applying subjective filtering, we use the government's own road classification as the authority on what constitutes an expressway.

Each LineString is decomposed into individual coordinate points. The resulting dataset provides dense, accurate expressway geometry sourced directly from official government data.

In [32]:
import json

with open('../data/raw/national_map_lines.geojson', 'r') as f:
    data = json.load(f)

# Get unique FOLDERPATH values
folder_paths = set()
for feature in data['features']:
    folder_paths.add(feature['properties']['FOLDERPATH'])

print("Unique FOLDERPATH values:")
for fp in sorted(folder_paths):
    print(fp)

Unique FOLDERPATH values:
Layers/Contour_250K
Layers/Expressway
Layers/Expressway_Sliproad
Layers/International_bdy
Layers/Major_Road


In [41]:
expressway_coords = []

for feature in data['features']:
    if feature['properties']['FOLDERPATH'] == 'Layers/Expressway':
        coords = feature['geometry']['coordinates']
        for coord in coords:
            expressway_coords.append({
                'longitude': coord[0],
                'latitude': coord[1],
                'name': feature['properties']['NAME']
            })

expressway_df = pd.DataFrame(expressway_coords)
print(f"Expressway points: {len(expressway_df):,}")
print(expressway_df['name'].value_counts())

expressway_df.to_csv('../data/processed/expressway_coords.csv', index=False)
print("\nSaved to data/processed/expressway_coords.csv")

Expressway points: 3,319
name
PAN ISLAND EXPRESSWAY            908
AYER RAJAH EXPRESSWAY            385
CENTRAL EXPRESSWAY               382
KALLANG PAYA LEBAR EXPRESSWAY    302
TAMPINES EXPRESSWAY              264
SELETAR EXPRESSWAY               262
EAST COAST PARKWAY               201
BUKIT TIMAH EXPRESSWAY           188
KRANJI EXPRESSWAY                166
MARINA COASTAL EXPRESSWAY        120
LORNIE HIGHWAY                    70
SIME UNDERPASS                    60
NICOLL UNDERPASS                   7
ADAM UNDERPASS                     4
Name: count, dtype: int64

Saved to data/processed/expressway_coords.csv


## Bus Stops
Bus stop data is downloaded from data.gov.sg as a GeoJSON file published by LTA. Coordinates are embedded in the GeoJSON geometry, so no geocoding is required.

We extract the bus stop code and coordinates for each of the 5,166 bus stops in the dataset.

In [44]:
import json

with open('../data/raw/bus_stops.geojson', 'r') as f:
    bus_data = json.load(f)

print(f"Type: {bus_data['type']}")
print(f"Number of features: {len(bus_data['features'])}")
print("\nFirst feature:")
print(bus_data['features'][0])

Type: FeatureCollection
Number of features: 5166

First feature:
{'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [103.82558173624862, 1.4465079238607317]}, 'properties': {'OBJECTID_1': 144151, 'BUS_STOP_NUM': '58039', 'BUS_ROOF_NUM': 'UNK', 'UNIQUE_ID': 148, 'INC_CRC': 'CF69435A8DF83116', 'FMEL_UPD_D': '20240107220422'}}


In [45]:
bus_stops = []
for feature in bus_data['features']:
    coords = feature['geometry']['coordinates']
    bus_stops.append({
        'bus_stop_num': feature['properties']['BUS_STOP_NUM'],
        'longitude': coords[0],
        'latitude': coords[1]
    })

bus_stops_df = pd.DataFrame(bus_stops)
print(f"Total bus stops: {len(bus_stops_df):,}")
bus_stops_df.head()

Total bus stops: 5,166


,bus_stop_num,longitude,latitude
0,58039,103.825582,1.446508
1,58031,103.825139,1.446489
2,14199,103.812726,1.281300
3,22259,103.702624,1.340913
4,66581,103.879735,1.382448


In [46]:
bus_stops_df.to_csv('../data/processed/bus_stops.csv', index=False)
print(f"Saved {len(bus_stops_df):,} bus stops to data/processed/bus_stops.csv")

Saved 5,166 bus stops to data/processed/bus_stops.csv
